# معالجة البيانات وبناء خط المعالجة (Data Preprocessing & Pipeline)

## مقدمة
في هذه المرحلة، سنقوم بتحويل البيانات الخام التي قمنا بتحليلها في الخطوة السابقة إلى تنسيق جاهز لتدريب نماذج تعلم الآلة. سنركز على التنظيم المنهجي لضمان عدم حدوث تسرب للبيانات (Data Leakage) وضمان قابلية إعادة استخدام الكود.

## ماذا سنفعل في هذا الـ Notebook؟
1. تحميل مجموعة البيانات.
2. فصل الميزات (Features) عن المتغير المستهدف (Target).
3. تقسيم البيانات إلى مجموعات تدريب واختبار مع الحفاظ على نسبة الفئات (Stratification).
4. بناء خط معالجة (Pipeline) لتوحيد مقاييس البيانات (Scaling).
5. تطبيق المعالجة بشكل صحيح علمياً.

## 2. تحميل البيانات (Load Dataset)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 3. فصل الميزات عن الهدف (Split X and y)

In [2]:
# فصل المتغير الهدف عن الميزات
X = df.drop("Class", axis=1)
y = df["Class"]

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (284807, 30)
Shape of y: (284807,)


### ملاحظة:
المتغير `Class` هو الهدف الذي نسعى للتنبؤ به (0 للعمليات السليمة، 1 للاحتيال). نقوم بفصله الآن لضمان عدم استخدامه في أي عملية معالجة للميزات، مما يمنع **Data Leakage**.

## 4. تقسيم البيانات (Train-Test Split) مع Stratify

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train class distribution:
Class
0    0.998271
1    0.001729
Name: proportion, dtype: float64

Test class distribution:
Class
0    0.99828
1    0.00172
Name: proportion, dtype: float64


### لماذا استخدمنا `stratify=y`؟
بما أن البيانات غير متوازنة بشكل حاد، فإن استخدام `stratify` يضمن أن نسبة العمليات الاحتيالية في مجموعة التدريب هي نفسها في مجموعة الاختبار، مما يجعل تقييم النموذج أكثر دقة وواقعية.

## 5. استراتيجية توحيد المقاييس (Scaling Strategy)

سنستخدم **StandardScaler** لتوحيد مقاييس الميزات. هذه الخطوة ضرورية لأن بعض الميزات مثل `Amount` و `Time` لها نطاقات قيم مختلفة تماماً عن الميزات المحولة (V1-V28)، مما قد يؤثر سلباً على أداء الخوارزميات التي تعتمد على المسافات.

## 6. بناء خط المعالجة (Build Pipeline)

In [4]:
# إنشاء Pipeline للـ Scaling فقط (بدون نموذج حالياً)
preprocessing_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

preprocessing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


## 7. تطبيق الـ Scaling (Fit على Train فقط)

In [5]:
# تطبيق الـ Scaling
X_train_scaled = preprocessing_pipeline.fit_transform(X_train)
X_test_scaled = preprocessing_pipeline.transform(X_test)

print("Scaled Train shape:", X_train_scaled.shape)
print("Scaled Test shape:", X_test_scaled.shape)

Scaled Train shape: (227845, 30)
Scaled Test shape: (56962, 30)


### نقطة أكاديمية هامة:
- **fit_transform** تم تطبيقها على مجموعة التدريب فقط ليتعلم الـ Scaler المتوسط والانحراف المعياري من بيانات التدريب.
- **transform** تم تطبيقها على مجموعة الاختبار باستخدام المعايير التي تعلمها من التدريب.
هذا يضمن أننا لا نستخدم أي معلومات من مجموعة الاختبار أثناء التدريب، وهو ما يسمى بمنع **Data Leakage**.

## 8. الخاتمة
لقد أتممنا بنجاح تجهيز البيانات وبناء خط المعالجة. البيانات الآن موزعة بشكل صحيح، موحدة المقاييس، وجاهزة تماماً للمرحلة القادمة وهي تدريب النماذج وتقييمها.